```yaml
title: Scoring federal accounts by contract share
description: Which agency accounts actually buy things from contractors — and why the biggest ones usually don't.
tags: [budget, shape, business-development]
endpoints: [list_budget_accounts]
```

# Scoring federal accounts by contract share

Most federal agency budgets are misleading proxies for real contracting opportunity. A topline like "HHS spent $2.5T last year" tells you almost nothing about where capture work pays off, because most of that money is transfers, not procurement.

Every Tango budget account row carries `contract_share_of_obligated_capped` — the fraction of obligated dollars that actually moved through contract awards. Sort by it and the universe collapses to the accounts worth your team's time.

## Setup

In [1]:
from tango import TangoClient

client = TangoClient()  # reads TANGO_API_KEY from env; or TangoClient(api_key="...")

## The misleading view

Start with the question a capture lead would actually ask: *which HHS accounts should my team be tracking?* The naive answer — "the biggest ones" — is the wrong answer.

HHS's agency code is `075`. Pull every FY24 account, sort by obligation size, look at the top three.

In [2]:
# HHS has ~185 accounts; page through them (server caps page size at 100)
def pull_all_accounts(agency_code, fiscal_year=2024):
    accounts, page = [], 1
    while True:
        r = client.list_budget_accounts(
            agency_code=agency_code, fiscal_year=fiscal_year, page=page, limit=100,
            shape=(
                "federal_account_symbol,account_title,agency_name,"
                "obligated_total,contract_obligated,"
                "contract_share_of_obligated_capped"
            ),
        )
        accounts.extend(r.results)
        if not r.next:
            return accounts
        page += 1


hhs = pull_all_accounts("075")
print(f"HHS accounts FY24: {len(hhs)}")

by_size = sorted(hhs, key=lambda a: float(a.get("obligated_total") or 0), reverse=True)
print("\nTop 3 HHS accounts by obligated_total:")
for a in by_size[:3]:
    obl = float(a.get("obligated_total") or 0) / 1e9
    share = a.get("contract_share_of_obligated_capped")
    share_str = f"{float(share):.1%}" if share is not None else "n/a"
    print(f"  ${obl:>6.0f}B  contract share {share_str:>5}  {a['account_title']}")

HHS accounts FY24: 185

Top 3 HHS accounts by obligated_total:
  $   690B  contract share  1.0%  Grants to States for Medicaid
  $   526B  contract share  0.0%  Federal Supplementary Medical Insurance Trust Fund
  $   503B  contract share   n/a  Payments to Health Care Trust Funds


The three biggest HHS accounts are ~$1.7T combined — and ~0% of that touches a contractor. Medicaid grants flow to states; Medicare trust funds pay providers; trust-fund transfers are intragovernmental. None of these are capture targets.

## The contractability lens

Now sort by `contract_share_of_obligated_capped` instead — and filter to accounts large enough to matter (≥$500M obligated) where contracts are the dominant motion (≥50% share). That's the threshold below which an account is either too small to staff against or too transfer-heavy to be a real procurement surface.

In [3]:
MIN_OBLIGATED = 500e6
MIN_SHARE = 0.50

def contracting_surface(accounts, min_obligated=MIN_OBLIGATED, min_share=MIN_SHARE):
    passed = [
        a for a in accounts
        if (a.get("contract_share_of_obligated_capped") is not None)
        and float(a["contract_share_of_obligated_capped"]) >= min_share
        and float(a.get("obligated_total") or 0) >= min_obligated
    ]
    passed.sort(key=lambda a: float(a["contract_obligated"] or 0), reverse=True)
    return passed


hhs_surface = contracting_surface(hhs)
print(f"HHS accounts that pass the contractability filter: {len(hhs_surface)} of {len(hhs)}\n")
for a in hhs_surface:
    contracts = float(a.get("contract_obligated") or 0) / 1e9
    share = float(a["contract_share_of_obligated_capped"])
    print(f"  ${contracts:>5.2f}B contracts  share={share:.0%}  {a['federal_account_symbol']}  {a['account_title']}")

HHS accounts that pass the contractability filter: 7 of 185

  $ 5.23B contracts  share=70%  075-0511  Program Management
  $ 4.03B contracts  share=54%  075-0140  Public Health and Social Services Emergency Fund
  $ 2.15B contracts  share=84%  075-1000  Administration for Strategic Preparedness and Response
  $ 1.61B contracts  share=58%  075-4554  Services and Supply Fund
  $ 1.41B contracts  share=55%  075-8393  Health Care Fraud and Abuse Control Account
  $ 1.07B contracts  share=91%  075-0519  Quality Improvement Organizations
  $ 0.56B contracts  share=71%  075-0522  Center for Medicare and Medicaid Innovation


That's the HHS contracting surface in a few lines. Program Management, ASPR, the Public Health emergency fund, the Services and Supply Fund, Quality Improvement Orgs — these are where a capture team should be spending its attention, and they don't show up anywhere on the topline HHS budget table.

## How concentrated is it?

Of the ~$37B HHS spent on contracts in FY24, how much lives in those few accounts?

In [4]:
total_contracts = sum(float(a.get("contract_obligated") or 0) for a in hhs)
filtered_contracts = sum(float(a.get("contract_obligated") or 0) for a in hhs_surface)

print(f"HHS contract obligations FY24: ${total_contracts/1e9:.1f}B")
print(f"  Captured by the {len(hhs_surface)} filtered accounts: ${filtered_contracts/1e9:.1f}B ({filtered_contracts/total_contracts:.0%})")
print(f"  Spread across the other {len(hhs)-len(hhs_surface)} accounts:        ${(total_contracts-filtered_contracts)/1e9:.1f}B ({1-filtered_contracts/total_contracts:.0%})")

HHS contract obligations FY24: $36.7B
  Captured by the 7 filtered accounts: $16.1B (44%)
  Spread across the other 178 accounts:        $20.7B (56%)


**Nota bene:** the filter is editorial. Lower the size cutoff and you pull in smaller specialty accounts (working capital funds, facilities accounts) that might be exactly what a niche vendor wants. Lower the contract-share threshold and you start including megaplex accounts where contracts ride alongside grants. The point isn't the specific cut — it's that one field on the API turns "the HHS budget" into a sortable, filterable list.

## Does the pattern generalize?

The trick isn't HHS-specific. Run the same filter against any agency. Here's the Department of Energy (`agency_code="089"`) — structurally different from HHS (DOE is contractor-heavy, HHS is transfer-heavy) but the filter tells the same story.

In [5]:
doe = pull_all_accounts("089")
doe_surface = contracting_surface(doe)

doe_total = sum(float(a.get("contract_obligated") or 0) for a in doe)
doe_filtered = sum(float(a.get("contract_obligated") or 0) for a in doe_surface)

print(f"DOE FY24: {len(doe_surface)} of {len(doe)} accounts pass — capture {doe_filtered/doe_total:.0%} of DOE contracting (${doe_filtered/1e9:.1f}B of ${doe_total/1e9:.1f}B)\n")
for a in doe_surface:
    contracts = float(a.get("contract_obligated") or 0) / 1e9
    share = float(a["contract_share_of_obligated_capped"])
    print(f"  ${contracts:>6.2f}B contracts  share={share:.0%}  {a['federal_account_symbol']}  {a['account_title']}")

DOE FY24: 7 of 69 accounts pass — capture 86% of DOE contracting ($41.4B of $48.2B)

  $ 21.44B contracts  share=91%  089-0240  Weapons Activities
  $  6.86B contracts  share=74%  089-0222  Science
  $  6.59B contracts  share=86%  089-0251  Defense Environmental Cleanup
  $  2.56B contracts  share=97%  089-0309  Defense Nuclear Nonproliferation
  $  1.78B contracts  share=96%  089-0314  Naval Reactors
  $  1.27B contracts  share=77%  089-0243  Other Defense Activities
  $  0.84B contracts  share=97%  089-5231  Uranium Enrichment Decontamination and Decommissioning Fund


Same shape, different scale. The Weapons Activities account alone runs ~$21B through contracts at a 91% contract share — one row on the API, one of the largest single contracting surfaces in the federal government, and invisible from a topline DOE summary that lumps it in with applied energy R&D and clean-energy loan programs.

## Bonus: where to go next

Once you've picked an account worth tracking, two natural follow-ups:

- **Who's winning at that account?** Call `get_budget_account_recipients(account_id)` and you get the primes and dollar amounts behind that line. (See the [Deep Space Exploration notebook](./budget-deep-space-exploration.ipynb) for the worked example — Boeing, SpaceX, Blue Origin, Lockheed, all in five lines of code.)
- **Is the account growing?** Add `ba_growth_next_year_pct` to the `shape` and re-sort. Accounts that pass the contractability filter *and* have rising budget authority are the highest-priority capture targets.

Both stay one API call away.